In [0]:
--------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`silver`.`pedidos_cdc` (
    pedido_id STRING,
    cliente_id STRING,
    produto_id STRING,
    quantidade BIGINT,
    valor_total DOUBLE,
    data_pedido TIMESTAMP
);
--------------------------------
SELECT * FROM `capgemini_academy`.`bronze`.`pedidos_cdc_raw`
ORDER BY pedido_id, op_ts;
--------------------------------
CREATE OR REPLACE TEMP VIEW v_cdc_latest AS
    SELECT *
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY pedido_id ORDER BY op_ts DESC) AS rn
        FROM `capgemini_academy`.`bronze`.`pedidos_cdc_raw`
    )
    WHERE rn = 1;
--------------------------------
SELECT * FROM v_cdc_latest;
--------------------------------
MERGE INTO `capgemini_academy`.`silver`.`pedidos_cdc` AS tgt
    USING v_cdc_latest AS src
        ON tgt.pedido_id = src.pedido_id
    WHEN MATCHED AND src.op = 'D'
        THEN DELETE
    WHEN MATCHED AND src.op = 'U'
        THEN UPDATE SET
            tgt.cliente_id = src.cliente_id,
            tgt.produto_id = src.produto_id,
            tgt.quantidade = src.quantidade,
            tgt.valor_total = src.valor_total,
            tgt.data_pedido = src.data_pedido
    WHEN NOT MATCHED AND src.op IN ('I','U')
        THEN INSERT (pedido_id, cliente_id, produto_id, quantidade, valor_total, data_pedido)
        VALUES (src.pedido_id, src.cliente_id, src.produto_id, src.quantidade, src.valor_total, src.data_pedido);
--------------------------------
SELECT * FROM `capgemini_academy`.`silver`.`pedidos_cdc`;
--------------------------------
CREATE OR REPLACE TABLE `capgemini_academy`.`gold`.`receita_total` AS
    SELECT
        COUNT(*) pedidos,
        SUM(valor_total) receita
    FROM `capgemini_academy`.`silver`.`pedidos_cdc`;
--------------------------------
SELECT * FROM `capgemini_academy`.`gold`.`receita_total`;
--------------------------------
